# **Practical Exercise and Solution**


- You are asked to write a Deep neural network to classify [IRIS](https://en.wikipedia.org/wiki/Iris_flower_data_set))? dataset.
  - IRIS dataset consists of 3 different types of irises (i.e., classes) and Four features such as Sepal Length, Sepal Width, Petal Length and Petal Width (number of input unit).
  - You can use the code that you see in Tutorial Parts 1 and 2.

### **1. Importing the required libraries**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from sklearn import datasets
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
%matplotlib inline

### **2. Loading the data**

  - First, we have to collect the IRIS dataset from Scikit-learn dataset pacakges using `load_iris()` function.

In [ ]:
iris = datasets.load_iris()

- Extracting data (X) and target (Y)

In [ ]:
data = iris['data']
target = iris['target']
data.shape, target.shape
type(data), type(target)

### **3. Normalising the data**

In [ ]:
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data)

### **4. Splitting data into train and test**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(data_scaled, target, test_size=0.2, random_state=2)
type(X_train), type(X_test), type(y_train), type(y_test)

### **5. Converting the numpy format data to tensor**

In [ ]:
X_train = torch.FloatTensor(X_train)
X_test = torch.FloatTensor(X_test)
y_train = torch.LongTensor(y_train)
y_test = torch.LongTensor(y_test)
type(X_train), type(X_test), type(y_train), type(y_test)

### **6. Encoding the target to one-hot for multi-class classification**

In [ ]:
y_train_one_hot = F.one_hot(y_train, num_classes=3)
y_test_one_hot = F.one_hot(y_test, num_classes=3)

### **7. Defining the classification network**

In [ ]:
class IRISClassifier(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(IRISClassifier, self).__init__()

        self.network = nn.Sequential(
        nn.Linear(input_dim, 128),
        nn.ReLU(),
        nn.Linear(128, 64),
        nn.Linear(64, output_dim), #nn.Softmax(dim=1)
        nn.LogSoftmax(dim=1)
        )

    def forward(self, x):
        return self.network(x)

### **8. Initialising the network**

In [ ]:
input_dim  = 4
output_dim = 3
learning_rate = 0.01
num_epochs = 200
model = IRISClassifier(input_dim, output_dim)

### **9. Choosing the loss function**

In [ ]:
#criterion = nn.CrossEntropyLoss() #because this is a classification problem (multi-class)
#criterion = nn.KLDivLoss(reduction='batchmean') #because this is a classification problem (multi-class)
criterion = nn.NLLLoss() #because this is a classification problem (multi-class)

### 10. **Choosing the optimiser**

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

### **11. Learning the model on train data**

In [ ]:
for epoch in range(num_epochs):
  output = model(X_train)
  #print (output)

  #computing the loss value
  #y_train_one_hot = F.one_hot(y_train, num_classes=3)
  #print (y_train_one_hot)
  #target = torch.log(y_train_one_hot)
  #print (y_train_one_hot.shape)
  #print (output.shape)
  #print (F.softmax(y_train_one_hot.float, dim=1))

  #loss = criterion(output, y_train_one_hot.long())

  loss = criterion(output, y_train)
  print ("Epoch: {}, Loss: {}".format(epoch, loss))

  #clear out the gradients from the last step loss.backward()
  optimizer.zero_grad()

  #backward propagation: calculate gradients
  loss.backward()

  #update the weights
  optimizer.step()

### **12. Evaluating the model on the test data**

In [ ]:
with torch.no_grad():
  output_test = model(X_test)
  #print (output_test)
  pred = output_test.detach().numpy()
  target = y_test.detach().numpy()

  pred_cls = np.argmax(pred, axis=1)
  #print (pred_cls)
  #print (target)
  acc = (pred_cls == target).sum()/pred_cls.shape[0]
  print ("Accuracy on test data: {}".format(acc))

### **13. Monitoring the loss and evaluation metrics**

- Defining a function to compute accuracy

In [ ]:
def get_accuracy(pred, actual):
    "Assuming that the arguments are tensor"
    y_pred = pred.detach().numpy()
    y_actual = actual.detach().numpy()

    pred_cls = np.argmax(y_pred, axis=1)
    acc = (pred_cls == y_actual).sum()/y_pred.shape[0]
    return acc

- Monitoring training loss, training accuracy and testing accuracy

In [ ]:
#keeping the train loss, train accuracy, test accuracy
train_loss = []
train_accuracy = []
test_accuracy = []

for epoch in range(num_epochs):
  X_train_pred = model(X_train)
  train_accuracy.append(get_accuracy(X_train_pred, y_train))

  #computing the loss value
  loss = criterion(X_train_pred, y_train)
  train_loss.append(loss.item())

  #clear out the gradients from the last step loss.backward()
  optimizer.zero_grad()

  #backward propagation: calculate gradients
  loss.backward()

  #update the weights
  optimizer.step()

  with torch.no_grad():
    X_test_pred = model(X_test)
    test_accuracy.append(get_accuracy(X_test_pred, y_test))

  if (epoch + 1) % 5 == 0:
    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {loss.item():.4f}, Train Accuracy: {sum(train_accuracy)/len(train_accuracy):.4f}, Test Accuracy: {sum(test_accuracy)/len(test_accuracy):.4f}")

### **14. Visualising the results**

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(3, figsize=(12, 6), sharex=True)

ax1.plot(train_loss)
ax1.set_ylabel("training loss", color='red')

ax2.plot(train_accuracy, color='green')
ax2.set_ylabel("training accuracy")

ax3.plot(test_accuracy)
ax3.set_ylabel("test accuracy", color='blue')

ax3.set_xlabel("epochs")